In [1]:
from dotenv import load_dotenv    
import os                         

load_dotenv()  

True

In [3]:
import yfinance as yf    

In [2]:
from crewai import Agent, Task, Crew, Process, LLM 
from crewai.tools import tool
                   


llm = LLM(
    model="openrouter/openai/gpt-4o-mini",            
    base_url="https://openrouter.ai/api/v1",   
    api_key=os.getenv("API_TOKEN")           
)

In [4]:
@tool("Market Data Fetcher")
def fetch_market_data(sector: str) -> str:
    """
    Fetches real market data for a sector using sector ETFs.
    Args:
        sector: The market sector (e.g., 'technology', 'healthcare', 'energy')
    """
    
    sector_etfs = {
        "technology": "XLK",
        "healthcare": "XLV",
        "energy": "XLE",
        "financial": "XLF",
        "consumer": "XLY",
        "industrial": "XLI",
        "real estate": "XLRE",
    }
    
    etf_symbol = sector_etfs.get(sector.lower(), "SPY")  
    
    try:
        etf = yf.Ticker(etf_symbol)
        hist = etf.history(period="3mo")          
        info = etf.info
        
        
        start_price = hist['Close'].iloc[0]
        end_price = hist['Close'].iloc[-1]
        change_pct = ((end_price - start_price) / start_price) * 100
        
        if change_pct > 5:
            trend = "Bullish"
        elif change_pct < -5:
            trend = "Bearish"
        else:
            trend = "Neutral"
        
        
        return f"""
        Sector: {sector.upper()} (ETF: {etf_symbol})
        Current Price: ${end_price:.2f}
        3-Month Change: {change_pct:.1f}%
        Trend: {trend}
        52-Week High: ${info.get('fiftyTwoWeekHigh', 'N/A')}
        52-Week Low: ${info.get('fiftyTwoWeekLow', 'N/A')}
        """
    except Exception as e:
        return f"Error fetching sector data: {str(e)}"

In [5]:
@tool("Stock Screener")
def screen_stocks(criteria: str) -> str:
    """
    Screens real stocks and returns live data.
    Args:
        criteria: What to screen for (e.g., 'top tech stocks')
    """
    
    stock_pools = {
        "tech": ["NVDA", "AAPL", "MSFT", "GOOGL", "META", "AMZN", "CRM", "PLTR", "CRWD", "AMD"],
        "healthcare": ["LLY", "UNH", "JNJ", "PFE", "ABBV", "MRK", "TMO", "ABT"],
        "energy": ["XOM", "CVX", "NEE", "FSLR", "ENPH", "SLB", "COP"],
    }
    
    
    if any(word in criteria.lower() for word in ["tech", "technology", "ai", "software"]):
        tickers = stock_pools["tech"]
    elif any(word in criteria.lower() for word in ["health", "pharma", "medical"]):
        tickers = stock_pools["healthcare"]
    elif any(word in criteria.lower() for word in ["energy", "oil", "solar"]):
        tickers = stock_pools["energy"]
    else:
        tickers = stock_pools["tech"]  
    
    output = f"Stock Screening Results for: {criteria}\n{'='*50}\n"
    
    for ticker in tickers:
        try:
            stock = yf.Ticker(ticker)
            info = stock.info
            
            price = info.get('currentPrice', info.get('regularMarketPrice', 'N/A'))
            pe = info.get('trailingPE', 'N/A')
            growth = info.get('revenueGrowth', 'N/A')
            if isinstance(growth, (int, float)):
                growth = f"{growth*100:.1f}%"
            market_cap = info.get('marketCap', 0)
            name = info.get('shortName', ticker)
            
            output += f"\n{ticker} - {name}"
            output += f"\n  Price: ${price} | P/E: {pe} | Revenue Growth: {growth}"
            output += f"\n  Market Cap: ${market_cap:,}"
            output += f"\n  52W Range: ${info.get('fiftyTwoWeekLow', 'N/A')} - ${info.get('fiftyTwoWeekHigh', 'N/A')}\n"
        except Exception as e:
            output += f"\n{ticker} - Error: {str(e)}\n"
    
    return output


In [6]:
@tool("Company Analyzer")
def analyze_company(ticker: str) -> str:
    """
    Deep-dive analysis of a specific company using real data.
    Args:
        ticker: Stock ticker symbol (e.g., AAPL, NVDA)
    """
    try:
        stock = yf.Ticker(ticker)
        info = stock.info
        
        output = f"\n{'='*50}\nDetailed Analysis: {ticker.upper()} - {info.get('shortName', 'N/A')}\n{'='*50}\n"
        output += f"  Sector: {info.get('sector', 'N/A')}\n"
        output += f"  Industry: {info.get('industry', 'N/A')}\n"
        output += f"  Current Price: ${info.get('currentPrice', info.get('regularMarketPrice', 'N/A'))}\n"
        output += f"  Market Cap: ${info.get('marketCap', 0):,}\n"
        output += f"  P/E Ratio: {info.get('trailingPE', 'N/A')}\n"
        output += f"  Forward P/E: {info.get('forwardPE', 'N/A')}\n"
        output += f"  Revenue: ${info.get('totalRevenue', 0):,}\n"
        output += f"  Revenue Growth: {info.get('revenueGrowth', 'N/A')}\n"
        output += f"  Profit Margin: {info.get('profitMargins', 'N/A')}\n"
        output += f"  Debt to Equity: {info.get('debtToEquity', 'N/A')}\n"
        output += f"  Free Cash Flow: ${info.get('freeCashflow', 0):,}\n"
        output += f"  52W High: ${info.get('fiftyTwoWeekHigh', 'N/A')}\n"
        output += f"  52W Low: ${info.get('fiftyTwoWeekLow', 'N/A')}\n"
        output += f"  Analyst Target: ${info.get('targetMeanPrice', 'N/A')}\n"
        output += f"  Recommendation: {info.get('recommendationKey', 'N/A')}\n"
        
        return output
    except Exception as e:
        return f"Error analyzing {ticker}: {str(e)}"

In [7]:
market_researcher = Agent(
    role="Senior Market Research Analyst",
    goal="Analyze current market conditions and identify promising sectors for investment",
    backstory="""You are a veteran market analyst with 20 years at JP Morgan.
    You have a keen eye for macro trends and sector rotation. You always
    start with the big picture before drilling into specifics. Your analysis
    has helped clients avoid major downturns and capitalize on bull runs.""",
    llm=llm,
    tools=[fetch_market_data],     
    verbose=True
)

In [8]:
stock_screener = Agent(
    role="Quantitative Stock Screener",
    goal="Screen and identify the most promising stock opportunities based on data-driven criteria",
    backstory="""You are a quant analyst who built screening algorithms at 
    Renaissance Technologies. You combine fundamental metrics (P/E, growth, 
    margins) with sector trends to find hidden gems. You never recommend a 
    stock without solid data backing.""",
    llm=llm,
    tools=[screen_stocks, analyze_company],   
    verbose=True
)

In [9]:
investment_advisor = Agent(
    role="Chief Investment Strategist",
    goal="Generate actionable investment recommendations with clear rationale and risk assessment",
    backstory="""You are the CIO of a boutique investment firm managing $500M.
    You are known for clear, actionable advice that balances risk and reward.
    You always include entry points, target prices, and stop-loss levels.
    You never make recommendations without considering downside risks.""",
    llm=llm,
    tools=[analyze_company],          
    verbose=True
)


In [10]:
market_research_task = Task(
    description="""Analyze the current market conditions for the {sector} sector.
    Use the Market Data Fetcher tool to get real data.
    
    Your analysis should cover:
    1. Overall sector trend (bullish/bearish/neutral)
    2. Key drivers and catalysts
    3. Major risks to watch
    4. Whether now is a good time to invest in this sector""",
    expected_output="""A comprehensive market analysis including:
    - Sector trend assessment
    - Key catalysts and drivers
    - Risk factors
    - Overall investment recommendation for the sector""",
    agent=market_researcher
)

In [11]:
screening_task = Task(
    description="""Based on the market research provided, screen for the best 
    stock opportunities in the {sector} sector.
    
    Use the Stock Screener tool to find candidates, then use the Company Analyzer 
    tool to do a deep dive on the top 2-3 picks.
    
    Focus on:
    - Strong revenue growth
    - Competitive advantages (moats)
    - Reasonable valuation relative to growth""",
    expected_output="""A ranked list of top 3 stock picks with:
    - Ticker, company name, current price
    - Key metrics (P/E, growth rate, margins)
    - Why this stock stands out
    - Potential risks""",
    agent=stock_screener,
    context=[market_research_task]    
)

In [ ]:
recommendation_task = Task(
    description="""Based on the market analysis and stock screening results, 
    create a professional investment recommendation report.
    
    For each recommended stock, include:
    1. Investment thesis (why buy)
    2. Entry price suggestion
    3. 12-month price target
    4. Stop-loss level
    5. Key risks and what could go wrong
    6. Portfolio allocation suggestion (% of portfolio)""",
    expected_output="""A professional investment recommendation report formatted as:
    
    INVESTMENT RECOMMENDATIONS - {sector} Sector
    ============================================
    
    For each stock:
    - Ticker & Name
    - Investment Thesis
    - Entry Price / Target / Stop-Loss
    - Risk Assessment
    - Allocation Suggestion
    
    Plus an overall portfolio strategy summary.""",
    agent=investment_advisor,
    context=[market_research_task, screening_task], 
    output_file="output/stock_recommendations.md"    
)

In [ ]:
stock_picker_crew = Crew(
    agents=[market_researcher, stock_screener, investment_advisor],
    tasks=[market_research_task, screening_task, recommendation_task],
    process=Process.sequential,       
    verbose=True
)

In [14]:
result = await stock_picker_crew.kickoff_async(
    inputs={"sector": "technology"}  
)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: e826ad19-ce1c-45f7-a0b6-ded6813dce27                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Analyze the current market conditions for the technology sector.                                         │
│      Use the Market Data Fetcher tool to get real data.                                                         │
│                                                                                                                 │
│      Your analysis should cover:                                                                                │
│      1. Overall sector trend (bullish/bearish/neutral)                                                          │
│      2. Key drivers and catalysts                                                                               │
│      3. Major risks to watch                                                                                    │
│      4. Whether now is a good time to invest in this sector                                                     │
│  ID: 1f84af05-3cf0-4cac-a3cf-f21b65f002ac                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Senior Market Research Analyst                                                                          │
│                                                                                                                 │
│  Task: Analyze the current market conditions for the technology sector.                                         │
│      Use the Market Data Fetcher tool to get real data.                                                         │
│                                                                                                                 │
│      Your analysis should cover:                                                                                │
│      1. Overall sector trend (bullish/bearish/neutral)                                                          │
│      2. Key drivers and catalysts                                                                               │
│      3. Major risks to watch                                                                                    │
│      4. Whether now is a good time to invest in this sector                                                     │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: market_data_fetcher                                                                                      │
│  Args: {'sector': 'technology'}                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool market_data_fetcher executed with result: 
        Sector: TECHNOLOGY (ETF: XLK)
        Current Price: $180.04
        3-Month Change: 32.5%
        Trend: Bullish
        52-Week High: $198.73
        52-Week Low: $126.6
        ...


...

Tool company_analyzer executed with result (from cache): 
Detailed Analysis: NVDA - NVIDIA Corporation
  Sector: Technology
  Industry: Semiconductors
  Cu...
Tool company_analyzer executed with result (from cache): 
Detailed Analysis: GOOGL - Alphabet Inc.
  Sector: Communication Services
  Industry: Internet Co...
Tool company_analyzer executed with result (from cache): 
Detailed Analysis: META - Meta Platforms, Inc.
  Sector: Communication Services
  Industry: Inter...
[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Chief Investment Strategist                                                                             │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  # INVESTMENT RECOMMENDATIONS - Technology Sector                                                               │
│  ============================================                                                                   │
│                                                                                                                 │
│  ## 1. NVIDIA Corporation (NVDA)                                                                                │
│  - **Investment Thesis:**                                                                                       │
│    NVIDIA is a frontrunner in the semiconductor industry, benefiting from the skyrocketing demand for GPUs due  │
│  to AI and gaming. With a robust revenue growth rate of 85.2% and an upcoming expansion in its data center      │
│  business, NVIDIA's dominance should continue, making it a strong buy.                                          │
│                                                                                                                 │
│  - **Entry Price / Target / Stop-Loss:**                                                                        │
│    - Entry Price: $193.61                                                                                       │
│    - 12-Month Target Price: $301.62                                                                             │
│    - Stop-Loss Level: $170.00 (10.9% downside)                                                                  │
│                                                                                                                 │
│  - **Risk Assessment:**                                                                                         │
│    - **Key Risks:** High valuation concerns may lead to increased volatility. Supply chain disruptions could    │
│  hinder production capabilities, impacting growth.                                                              │
│                                                                                                                 │
│  - **Allocation Suggestion:**                                                                                   │
│    30% of the portfolio.                                                                                        │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ## 2. Alphabet Inc. (GOOGL)                                                                                    │
│  - **Investment Thesis:**                                                                                       │
│    Alphabet is well-positioned in online advertising and cloud services, supported by a diverse product         │
│  ecosystem and ongoing growth in AI investments. The projected revenue growth of 21.8% underlines its           │
│  resilience even in economic downturns, making it a compelling buy.                                             │
│                                                                                                                 │
│  - **Entry Price / Target / Stop-Loss:**               

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Based on the market analysis and stock screening results,                                                │
│      create a professional investment recommendation report.                                                    │
│                                                                                                                 │
│      For each recommended stock, include:                                                                       │
│      1. Investment thesis (why buy)                                                                             │
│      2. Entry price suggestion                                                                                  │
│      3. 12-month price target                                                                                   │
│      4. Stop-loss level                                                                                         │
│      5. Key risks and what could go wrong                                                                       │
│      6. Portfolio allocation suggestion (% of portfolio)                                                        │
│  Agent: Chief Investment Strategist                                                                             │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: e826ad19-ce1c-45f7-a0b6-ded6813dce27                                                                       │
│  Final Output: # INVESTMENT RECOMMENDATIONS - Technology Sector                                                 │
│  ============================================                                                                   │
│                                                                                                                 │
│  ## 1. NVIDIA Corporation (NVDA)                                                                                │
│  - **Investment Thesis:**                                                                                       │
│    NVIDIA is a frontrunner in the semiconductor industry, benefiting from the skyrocketing demand for GPUs due  │
│  to AI and gaming. With a robust revenue growth rate of 85.2% and an upcoming expansion in its data center      │
│  business, NVIDIA's dominance should continue, making it a strong buy.                                          │
│                                                                                                                 │
│  - **Entry Price / Target / Stop-Loss:**                                                                        │
│    - Entry Price: $193.61                                                                                       │
│    - 12-Month Target Price: $301.62                                                                             │
│    - Stop-Loss Level: $170.00 (10.9% downside)                                                                  │
│                                                                                                                 │
│  - **Risk Assessment:**                                                                                         │
│    - **Key Risks:** High valuation concerns may lead to increased volatility. Supply chain disruptions could    │
│  hinder production capabilities, impacting growth.                                                              │
│                                                                                                                 │
│  - **Allocation Suggestion:**                                                                                   │
│    30% of the portfolio.                                                                                        │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ## 2. Alphabet Inc. (GOOGL)                                                                                    │
│  - **Investment Thesis:**                                                                                       │
│    Alphabet is well-positioned in online advertising and cloud services, supported by a diverse product         │
│  ecosystem and ongoing growth in AI investments. The projected revenue growth of 21.8% underlines its           │
│  resilience even in economic downturns, making it a compelling buy.                                             │
│                                                                                                                 │
│  - **Entry Price / Target / Stop-Loss:**              

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [15]:
print("\n" + "=" * 60)
print("FINAL INVESTMENT RECOMMENDATIONS")
print("=" * 60)
print(result.raw)


FINAL INVESTMENT RECOMMENDATIONS
# INVESTMENT RECOMMENDATIONS - Technology Sector

## 1. NVIDIA Corporation (NVDA)
- **Investment Thesis:**  
  NVIDIA is a frontrunner in the semiconductor industry, benefiting from the skyrocketing demand for GPUs due to AI and gaming. With a robust revenue growth rate of 85.2% and an upcoming expansion in its data center business, NVIDIA's dominance should continue, making it a strong buy.

- **Entry Price / Target / Stop-Loss:**  
  - Entry Price: $193.61  
  - 12-Month Target Price: $301.62  
  - Stop-Loss Level: $170.00 (10.9% downside)

- **Risk Assessment:**  
  - **Key Risks:** High valuation concerns may lead to increased volatility. Supply chain disruptions could hinder production capabilities, impacting growth.

- **Allocation Suggestion:**  
  30% of the portfolio.

---

## 2. Alphabet Inc. (GOOGL)
- **Investment Thesis:**  
  Alphabet is well-positioned in online advertising and cloud services, supported by a diverse product ecosystem and 